# AXI host loop through an HTTP sidecar in front of a real vLLM server
A check of the plumbing, not a new result about AXI. AXI is a layer on the host side (architecture by Rajnish Choubey); the model and the engine are unchanged. This notebook starts a **real vLLM 0.31.0 server** with a small model, puts the repository's **sidecar** (`python/axi/sidecar.py`, standard library only) in front of it, drives it with the repository's own request stream for 200 steps, and counts on how many steps the state the host holds equals the oracle's state.

Needs a GPU runtime (Runtime > Change runtime type > a GPU). About 10 to 15 minutes, mostly the install. No key, no cost.
Limits: one small model, greedy decoding, 200 steps, synthetic service graphs, one run. The result is printed under the last cell and saved to `/content/sidecar_check_result.json`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install vllm==0.31.0

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 -b host-wrapper-sidecar https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/sidecar.py") and os.path.exists("/content/axi/python/axi/experiments/sidecar_check.py"), "the sidecar is not on GitHub (branch host-wrapper-sidecar) yet"
print("code ok")

In [ ]:
import subprocess, sys, time, urllib.request
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
logf = open("/content/vllm_server.log", "w")
server = subprocess.Popen([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--port", "8000", "--max-model-len", "4096",
                           "--gpu-memory-utilization", "0.85", "--dtype", "half", "--enable-prefix-caching"], stdout=logf, stderr=subprocess.STDOUT)
for i in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2); print("server up after about", i * 5, "s"); break
    except Exception: time.sleep(5)
else:
    print(open("/content/vllm_server.log").read()[-3000:]); raise SystemExit("the vLLM server did not start")

In [ ]:
import json, urllib.error
sys.path.insert(0, "/content/axi/python")
from axi.experiments.provider_side_constrained import REGEX      # the reply grammar the repository's other tests use
def probe(extra):
    body = dict(model=MODEL, messages=[{"role": "user", "content": "Reply with exactly: ADD[1,1:2]"}], max_tokens=32, temperature=0, **extra)
    req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    try: return json.load(urllib.request.urlopen(req, timeout=60))["choices"][0]["message"]["content"]
    except urllib.error.HTTPError as e: return "HTTP %d %s" % (e.code, e.read()[:300])
EXTRA = None
for name, extra in [("structured_outputs", {"structured_outputs": {"regex": REGEX}}), ("guided_regex", {"guided_regex": REGEX})]:
    out = probe(extra); print(name, "->", out)
    if not out.startswith("HTTP") and EXTRA is None: EXTRA = extra
print("constrained decoding field used:", list(EXTRA) if EXTRA else "NONE (the server accepted neither; steps below run unconstrained)")

In [ ]:
import threading
from axi.host import http_chat
from axi.sidecar import Sidecar, make_server
from axi.experiments import sidecar_check as SC
app = Sidecar(http_chat("http://127.0.0.1:8000/v1", MODEL, max_tokens=128, **(EXTRA or {})))
sidecar = make_server(app, "127.0.0.1", 8100); threading.Thread(target=sidecar.serve_forever, daemon=True).start()
STEPS = 200
res = SC.run("http://127.0.0.1:8100", STEPS)
res.update(model=MODEL, engine="vLLM 0.31.0", constrained=bool(EXTRA), steps_asked=STEPS)
json.dump(res, open("/content/sidecar_check_result.json", "w"), indent=1)
print("RESULT", json.dumps(res))
print(f"held state exactly the oracle's: {res['exact']} of {res['steps']} steps | replies read by the strict parser: {res['parsed']} | corrupted steps: {res['corrupted']}")
print(f"step seconds first/last tenth: {res['seconds_first_tenth']} / {res['seconds_last_tenth']} | prompt tokens first/last tenth: {res['prompt_tokens_first_tenth']} / {res['prompt_tokens_last_tenth']}")

Reading the result: the figures are for this one run (one model, greedy, 200 steps, synthetic graphs). They say the sidecar and the loop work end to end against a real engine; they do not say anything about other models or engines. Compare with the repository's test pages, which used the engine's own Python API on 100-step jobs.